# Web Usage

Membaca Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd

# Path dataset
file_path = "/content/drive/My Drive/dataset/NASA.csv"

df = pd.read_csv(file_path)

# Convert time
df["Timestamp"] = pd.to_datetime(df["time"], unit="s")

# Rename
df.rename(columns={"host": "Host"}, inplace=True)

df.head()


,Unnamed: 0,Host,time,method,url,response,bytes,Timestamp
0,0,***.novo.dk,805465029,GET,/ksc.html,200,7067,1995-07-11 12:17:09
1,1,***.novo.dk,805465031,GET,/images/ksclogo-medium.gif,200,5866,1995-07-11 12:17:11
2,2,***.novo.dk,805465051,GET,/images/MOSAIC-logosmall.gif,200,363,1995-07-11 12:17:31
3,3,***.novo.dk,805465053,GET,/images/USA-logosmall.gif,200,234,1995-07-11 12:17:33
4,4,***.novo.dk,805465054,GET,/images/NASA-logosmall.gif,200,786,1995-07-11 12:17:34


Konverai Angka ke Huruf

In [3]:
def num_to_letters(n):
    """0→A, 1→B, ..., 25→Z, 26→AA, dst."""
    result = ""
    n += 1
    while n > 0:
        n -= 1
        result = chr(65 + (n % 26)) + result
        n //= 26
    return result

Proses Web Usage dgn Waktu 20mnt/sesi

In [11]:
def process_web_usage(df):
    """
    - Filter hanya URL .html
    - Membuat label huruf untuk setiap halaman
    - Membentuk sesi 20 menit per Host
    """

    # --- Filter .html ---
    df = df[df["url"].str.endswith(".html")].copy()

    # --- Buat mapping URL → huruf ---
    unique_urls = df["url"].unique()
    url_map = {url: num_to_letters(i) for i, url in enumerate(unique_urls)}
    df["URL_Label"] = df["url"].map(url_map)

    # --- Sortir ---
    df = df.sort_values(by=["Host", "Timestamp"])

    results = []

    # --- Proses sesi per host ---
    for host, group in df.groupby("Host"):
        group = group.sort_values("Timestamp").reset_index(drop=True)

        sessions = []
        cur_session = []
        prev_time = None

        for _, row in group.iterrows():
            ts = row["Timestamp"]
            label = row["URL_Label"]

            if prev_time is None:
                cur_session.append((ts, label))
            else:
                delta = (ts - prev_time).total_seconds() / 60

                # sesi baru jika >20 menit
                if delta > 20:
                    sessions.append(cur_session)
                    cur_session = [(ts, label)]
                else:
                    cur_session.append((ts, label))

            prev_time = ts

        # simpan sesi terakhir
        if cur_session:
            sessions.append(cur_session)

        results.append({
            "Host": host,
            "Sessions": sessions
        })

    return results, url_map, df

In [12]:
url_mapping_df = pd.DataFrame([
    {"URL": url, "Label": label}
    for url, label in url_map.items()
])

url_mapping_df


,URL,Label
0,/ksc.html,A
1,/shuttle/missions/missions.html,B
2,/shuttle/resources/orbiters/columbia.html,C
3,/shuttle/missions/sts-69/mission-sts-69.html,D
4,/shuttle/countdown/liftoff.html,E
...,...,...
107,/shuttle/missions/sts-70/woodpecker.html,DD
108,/shuttle/technology/sts-newsref/spacelab.html,DE
109,/persons/nasa-cm/jmd.html,DF
110,/history/apollo/apollo-4/apollo-4.html,DG


In [7]:
table_rows = []

for item in results:
    host = item["Host"]
    sessions = item["Sessions"]

    for session_idx, session in enumerate(sessions, start=1):
        for ts, label in session:
            table_rows.append({
                "Host": host,
                "Session": session_idx,
                "Timestamp": ts,
                "Page": label
            })

usage_table = pd.DataFrame(table_rows)

usage_table


,Host,Session,Timestamp,Page
0,***.novo.dk,1,1995-07-11 12:17:09,A
1,***.novo.dk,1,1995-07-11 12:17:48,B
2,***.novo.dk,1,1995-07-11 12:23:01,C
3,***.novo.dk,2,1995-08-09 07:02:48,D
4,***.novo.dk,2,1995-08-09 07:05:38,E
...,...,...,...,...
643512,zzz.pe.u-tokyo.ac.jp,2,1995-07-13 11:04:06,BG
643513,zzz.pe.u-tokyo.ac.jp,2,1995-07-13 11:04:40,AL
643514,zzz.pe.u-tokyo.ac.jp,2,1995-07-13 11:14:51,AR
643515,zzz.pe.u-tokyo.ac.jp,2,1995-07-13 11:17:46,V


Hasil Sesi/Host

In [8]:
session_summary = usage_table.groupby("Host")["Session"].nunique().reset_index()
session_summary.rename(columns={"Session": "Total Sessions"}, inplace=True)

session_summary


,Host,Total Sessions
0,***.novo.dk,2
1,001.msy4.communique.net,1
2,007.thegap.com,2
3,01-dynamic-c.wokingham.luna.net,5
4,01.ts01.zircon.net.au,1
...,...,...
114786,zzsbtafe.slip.cc.uq.oz.au,1
114787,zzsmiege.slip.cc.uq.oz.au,1
114788,zztduffy.slip.cc.uq.oz.au,1
114789,zzz.pe.u-tokyo.ac.jp,2


Cetak Hasil

In [9]:
from openpyxl import Workbook

# Buat workbook baru
wb = Workbook()


# Sheet 1: URL Mapping

ws1 = wb.active
ws1.title = "URL Mapping"

# Header
ws1.append(list(url_mapping_df.columns))

# Isi data
for row in url_mapping_df.itertuples(index=False):
    ws1.append(list(row))


# Sheet 2: Web Usage Sessions

ws2 = wb.create_sheet("Web Usage Sessions")

ws2.append(list(usage_table.columns))
for row in usage_table.itertuples(index=False):
    ws2.append(list(row))


# Sheet 3: Session Summary

ws3 = wb.create_sheet("Session Summary")

ws3.append(list(session_summary.columns))
for row in session_summary.itertuples(index=False):
    ws3.append(list(row))

# Save file
output_path = "/content/web_usage_complete.xlsx"
wb.save(output_path)

print("✔ File Excel lengkap berhasil dibuat:")
print(output_path)


✔ File Excel lengkap berhasil dibuat:
/content/web_usage_complete.xlsx


Tabel Binary

In [10]:
# 5. MEMBANGUN TABEL BINARY PER HALAMAN (A=1, lainnya=0)

def build_binary_table(results, url_map):
    all_labels = sorted(url_map.values())  # A, B, C, ...
    rows = []

    for host_data in results:
        host = host_data["Host"]

        for session in host_data["Sessions"]:
            for ts, label in session:

                # buat baris kosong (semua 0)
                row = {l: 0 for l in all_labels}

                # halaman yang dikunjungi = 1
                row[label] = 1

                # tambahkan kolom lain
                row["Time"] = ts.strftime("%H:%M")
                row["Host"] = host

                rows.append(row)

    return pd.DataFrame(rows)


binary_table = build_binary_table(results, url_map)

print("=== Tabel Binary Halaman Dikunjungi ===")
display(binary_table.head(20))

# Simpan CSV bila diperlukan
binary_table.to_csv("binary_web_usage.csv", index=False)


=== Tabel Binary Halaman Dikunjungi ===


,A,AA,AB,AC,AD,AE,AF,AG,AH,AI,...,S,T,U,V,W,X,Y,Z,Time,Host
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:23,***.novo.dk
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,07:02,***.novo.dk
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,07:05,***.novo.dk
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,07:07,***.novo.dk
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,06:55,001.msy4.communique.net
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,21:24,007.thegap.com
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,21:28,007.thegap.com
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,21:37,007.thegap.com
